# Generate Gemma Codenames embeddings on a Colab GPU
Upload this notebook to https://colab.research.google.com/ (File → Upload notebook).
Choose **Runtime → Change runtime type → GPU**, then run cells in order.

You also need `gemma_colab_bundle.zip`, supplied alongside this notebook. It contains the current board/clue CSVs and a minimal exporter, **no credentials**.

Output: one `gemma_embeddings.npz` file containing both vocabularies, normalized vectors, and the exact model revision/STS prompt. This uses **google/embeddinggemma-2**, not Gemini API embeddings.

The local Colab disk keeps completed batches while the runtime exists. Rerun after a cell interruption to resume. A deleted/reset runtime loses that disk; the optional checkpoint cells below let you save and restore progress. No exact runtime estimate is promised; the encoder shows ETA after completing its first batch.

In [ ]:
%pip install -q "numpy>=2.1,<3" "sentence-transformers[image]>=6.1,<7" "transformers>=5.18,<6" "torch>=2.9,<3"
import torch
assert torch.cuda.is_available(), "Select a GPU runtime before continuing."
print("GPU:", torch.cuda.get_device_name(0))


## Upload the bundle
Select `gemma_colab_bundle.zip` when prompted. Rebuild this bundle locally after changing the CSVs.

In [ ]:
from google.colab import files
from pathlib import Path
import io, zipfile
uploaded = files.upload()
assert "gemma_colab_bundle.zip" in uploaded, "Upload gemma_colab_bundle.zip"
ROOT = Path("/content/gemma-codenames")
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(uploaded["gemma_colab_bundle.zip"])) as archive:
    for member in archive.infolist():
        destination = (ROOT / member.filename).resolve()
        assert destination.is_relative_to(ROOT.resolve()), "Unsafe archive path"
    archive.extractall(ROOT)
del uploaded
print("Bundle extracted to", ROOT)


## Optional Hugging Face authentication
Only needed if model access requires it. Add `HF_TOKEN` in Colab's Secrets panel (key icon), grant this notebook access, then run this cell. Do not paste tokens into notebook code. If access is gated, accept the model terms on Hugging Face yourself first.

In [ ]:
import os
from google.colab import userdata
try:
    token = userdata.get("HF_TOKEN")
except userdata.SecretNotFoundError:
    token = None
if token:
    os.environ["HF_TOKEN"] = token
    del token
    print("HF token configured without displaying it.")
else:
    print("No HF_TOKEN secret set; trying public model access.")


## Generate and validate
Default batch size is 128, using float32 and the same STS settings as the local backend. If CUDA runs out of memory, change it to 32 or 64 and rerun. Completed batches remain cached. The model revision is pinned to its resolved commit for this run.

Do not select synthetic mode: this exporter only generates real Gemma embeddings.

In [ ]:
import subprocess, sys
BATCH_SIZE = 128
command = [sys.executable, "-u", "-m", "gemma_codenames.export_embeddings",
    "--board", "gemma_codenames/data/board_words.csv",
    "--clues", "gemma_codenames/data/clue_words.csv",
    "--device", "cuda", "--batch-size", str(BATCH_SIZE),
    "--cache-dir", "/content/gemma-cache",
    "--output", "/content/gemma_embeddings.npz"]
# Pipe both streams into notebook output so the real error remains visible.
with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
if return_code:
    raise RuntimeError(
        f"Embedding exporter failed (exit {return_code}). "
        "The actual error is printed immediately above. "
        "Keep this runtime: completed embedding batches remain cached."
    )


## Download the single completed file

In [ ]:
from google.colab import files
files.download("/content/gemma_embeddings.npz")


## Use it on your Mac
1. Move the downloaded file into your project's `.cache/gemma_embeddings.npz`.
2. Add `GEMMA_EMBEDDINGS_FILE=.cache/gemma_embeddings.npz` to your local `.env`.
3. Restart with `./run.sh` and select real Gemma in the laboratory (do not launch with `--demo`).

The backend reads vectors directly without loading model weights or using a GPU. Words absent from the file are computed locally on demand with the same pinned Gemma revision and saved in the incremental cache. The first missing word may download/load Gemma; cached words do not load it. You may also extend the portable file in Colab. Keep the exported model identity and prompt intact. Both bots use this same file.

After downloading the result, disconnect/delete the Colab runtime to stop consuming GPU compute.

## Optional: save progress before deleting the runtime
If generation was interrupted, run this cell to download a consistent checkpoint of completed batches. The model files are not included. On a fresh runtime, run setup/upload cells, restore this checkpoint using the next cell, then run generation. The same model revision and software settings are needed for cache reuse.

In [ ]:
import sqlite3
from pathlib import Path
source = Path("/content/gemma-cache/embeddings/words.sqlite3")
assert source.exists(), "No embedding cache yet."
with sqlite3.connect(source) as src, sqlite3.connect("/content/gemma_checkpoint.sqlite3") as dst:
    src.backup(dst)
files.download("/content/gemma_checkpoint.sqlite3")


## Optional: restore a checkpoint BEFORE starting generation
Use only a checkpoint you generated. This refuses to overwrite an existing runtime cache.

In [ ]:
from google.colab import files
from pathlib import Path
checkpoint = files.upload()
assert "gemma_checkpoint.sqlite3" in checkpoint
cache = Path("/content/gemma-cache/embeddings/words.sqlite3")
cache.parent.mkdir(parents=True, exist_ok=True)
with cache.open("xb") as stream:
    stream.write(checkpoint["gemma_checkpoint.sqlite3"])
del checkpoint
print("Checkpoint restored. Run the generation cell.")
